<a href="https://colab.research.google.com/github/vince-fabmob/open_data_extractions/blob/main/bixi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pandas requests

import io
import re
import zipfile
import requests
import pandas as pd

In [ ]:
CKAN_BASE = "https://donnees.montreal.ca/api/3/action"
DATASET_ID = "bixi-historique-des-deplacements"

url = f"{CKAN_BASE}/package_show"
response = requests.get(url, params={"id": DATASET_ID}, timeout=30)
response.raise_for_status()

data = response.json()
data["success"], data.keys()

(True, dict_keys(['help', 'success', 'result']))

In [ ]:
resources = data["result"]["resources"]

df_resources = pd.DataFrame([
    {
        "name": r.get("name"),
        "format": r.get("format"),
        "url": r.get("url")
    }
    for r in resources
])

df_resources

,name,format,url
0,Déplacements BIXI - par mois,CSV,https://bixi.com/fr/donnees-ouvertes/


In [ ]:
def infer_year(text):
    m = re.search(r"(20\d{2})", str(text))
    return int(m.group(1)) if m else None

def count_rows_in_csv_bytes(raw_bytes):
    for enc in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
        try:
            df = pd.read_csv(io.BytesIO(raw_bytes), encoding=enc)
            return len(df)
        except Exception:
            pass
    raise ValueError("Impossible de lire le CSV")

def count_rows_from_zip_bytes(raw_bytes):
    total = 0
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as z:
        for name in z.namelist():
            if name.lower().endswith(".csv"):
                with z.open(name) as f:
                    total += count_rows_in_csv_bytes(f.read())
    return total

In [ ]:
yearly_counts = []

for r in resources:
    name = r.get("name", "")
    fmt = (r.get("format") or "").lower()
    file_url = r.get("url", "")

    year = infer_year(name + " " + file_url)

    if fmt not in ("csv", "zip"):
        continue
    if year is None:
        continue

    print(f"Traitement {year} - {name}")

    resp = requests.get(file_url, timeout=120)
    resp.raise_for_status()

    if fmt == "zip" or file_url.lower().endswith(".zip"):
        trips = count_rows_from_zip_bytes(resp.content)
    else:
        trips = count_rows_in_csv_bytes(resp.content)

    yearly_counts.append({
        "year": year,
        "trips": trips,
        "resource_name": name
    })

df_yearly = pd.DataFrame(yearly_counts)
df_yearly

""


In [ ]:
df_final = (
    df_yearly.groupby("year", as_index=False)["trips"]
    .sum()
    .sort_values("year")
)

df_final

KeyError: 'year'